# Predikcija kategorije proizvoda na osnovu naslova

Cilj projekta je razvoj modela mašinskog učenja koji na osnovu naziva proizvoda predviđa odgovarajuću kategoriju proizvoda.

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## Učitavanje i pregled podataka

Učitavam skup podataka i proveravam njegovu strukturu, kolone, nedostajuće vrednosti i raspodelu kategorija.

In [ ]:
df = pd.read_csv("products.csv")

print("Prvih 5 redova:")
display(df.head())

print("\nDimenzije skupa:")
print(df.shape)

print("\nNazivi kolona:")
print(df.columns)

print("\nNedostajuće vrednosti:")
print(df.isnull().sum())

## Čišćenje podataka

Pre modeliranja uklanjam nepotrebne razmake iz naziva kolona i redove kojima nedostaje naziv proizvoda ili kategorija. Takođe proveravam kategorije nakon čišćenja.

In [ ]:
# Uklanjanje nepotrebnih razmaka iz naziva kolona
df.columns = df.columns.str.strip()

# Uklanjanje redova bez naziva proizvoda ili kategorije
df = df.dropna(subset=["Product Title", "Category Label"])

# Standardizacija teksta
df["Product Title"] = df["Product Title"].str.lower().str.strip()
df["Category Label"] = df["Category Label"].str.strip()

# Provera nakon čišćenja
print("Dimenzije nakon čišćenja:")
print(df.shape)

print("\nNedostajuće vrednosti:")
print(df[["Product Title", "Category Label"]].isnull().sum())

print("\nBroj kategorija:")
print(df["Category Label"].nunique())

print("\nBroj proizvoda po kategoriji:")
print(df["Category Label"].value_counts())

## Standardizacija kategorija

Pregled kategorija pokazuje da postoje različiti nazivi za iste grupe proizvoda. Standardizujem ove vrednosti kako bi svaka kategorija imala jedinstven naziv.

In [ ]:
# Spajanje različitih naziva istih kategorija
df["Category Label"] = df["Category Label"].replace({
    "fridge": "Fridges",
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones"
})

# Provera kategorija nakon standardizacije
print("Broj kategorija nakon standardizacije:")
print(df["Category Label"].nunique())

print("\nBroj proizvoda po kategoriji:")
print(df["Category Label"].value_counts())

## Inženjering karakteristika

Pored samog teksta naslova, izdvajam nekoliko jednostavnih karakteristika koje mogu biti korisne za analizu: dužinu naslova, broj reči i prisustvo brojeva. Ove karakteristike mogu pokazati da li struktura naziva proizvoda nosi dodatne informacije o kategoriji.

In [ ]:
# Kreiranje dodatnih karakteristika iz naziva proizvoda
df["title_length"] = df["Product Title"].str.len()
df["word_count"] = df["Product Title"].str.split().str.len()
df["has_number"] = df["Product Title"].str.contains(r"\d").astype(int)

# Pregled novih karakteristika
display(
    df[
        ["Product Title", "Category Label",
         "title_length", "word_count", "has_number"]
    ].head()
)

print("\nProsečne vrednosti po kategoriji:")
display(
    df.groupby("Category Label")[
        ["title_length", "word_count", "has_number"]
    ].mean().round(2)
)

### Zapažanje o dodatnim karakteristikama

Dodatne karakteristike pokazuju određene razlike između kategorija. Na primer, CPU proizvodi u proseku imaju duže naslove i veći broj reči. Prisustvo brojeva je veoma često u skoro svim kategorijama, pa verovatno neće biti dovoljno jako kao samostalna karakteristika.

Za osnovni model koristiću tekst naslova sa TF-IDF vektorizacijom, a rezultate ću uporediti na više algoritama.

## Podela podataka i TF-IDF vektorizacija

Podatke delim na trening i test skup. Tekstualne naslove zatim pretvaram u numerički oblik pomoću TF-IDF vektorizacije.

In [ ]:
# Ulazni i ciljni podaci
X = df["Product Title"]
y = df["Category Label"]

# Podela na trening i test skup
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF vektorizacija
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Trening skup:", X_train_tfidf.shape)
print("Test skup:", X_test_tfidf.shape)

## Treniranje i poređenje modela

Za klasifikaciju proizvoda upoređujem dva različita algoritma: Logistic Regression i Linear SVC. Oba modela treniram nad istim podacima kako bih mogla direktno da uporedim njihove rezultate.

In [ ]:
# Modeli za poređenje
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC": LinearSVC()
}

# Treniranje i evaluacija modela
for name, model in models.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    model.fit(X_train_tfidf, y_train)

    y_pred = model.predict(X_test_tfidf)

    print("Accuracy:")
    print(accuracy_score(y_test, y_pred))

    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=model.classes_
    )

    disp.plot(xticks_rotation=90)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

### Poređenje rezultata

Oba modela daju veoma dobre rezultate, ali Linear SVC postiže veću tačnost od oko 97.2%, dok Logistic Regression postiže oko 96%.

Linear SVC ima bolje rezultate i kod većine pojedinačnih kategorija. Najviše grešaka se pojavljuje kod sličnih kategorija kao što su Fridges i Fridge Freezers.

Na osnovu dobijenih rezultata, Linear SVC biram kao finalni model.

## Treniranje i čuvanje finalnog modela

Linear SVC se pokazao kao najbolji model, pa ga treniram nad svim dostupnim podacima. Model i TF-IDF vektorizator čuvam kako bi kasnije mogli da se koriste za predikciju novih proizvoda.

In [ ]:
# TF-IDF nad svim dostupnim podacima
final_vectorizer = TfidfVectorizer()
X_tfidf = final_vectorizer.fit_transform(X)

# Treniranje finalnog modela
final_model = LinearSVC()
final_model.fit(X_tfidf, y)

# Čuvanje modela i vektorizatora
with open("product_category_model.pkl", "wb") as file:
    pickle.dump({
        "model": final_model,
        "vectorizer": final_vectorizer
    }, file)

print("Finalni model je uspešno treniran i sačuvan.")

## Testiranje finalnog modela

Finalni model testiram na nekoliko novih naziva proizvoda kako bih proverila da li predviđa očekivane kategorije.

In [ ]:
# Test proizvodi iz zadatka
test_products = [
    "iphone 7 32gb gold,4,3,Apple iPhone 7 32GB",
    "olympus e m10 mark iii geh use silber",
    "kenwood k20mss15 solo",
    "bosch wap28390gb 8kg 1400 spin",
    "bosch serie 4 kgv39vl31g",
    "smeg sbs8004po"
]

expected_categories = [
    "Mobile Phones",
    "Digital Cameras",
    "Microwaves",
    "Washing Machines",
    "Fridge Freezers",
    "Fridge Freezers"
]

test_vectors = final_vectorizer.transform(test_products)
predictions = final_model.predict(test_vectors)

results = pd.DataFrame({
    "Product Title": test_products,
    "Expected": expected_categories,
    "Predicted": predictions
})

display(results)

### Zapažanje nakon ručnog testiranja

Model je tačno klasifikovao 4 od 6 test proizvoda. Greške su se pojavile kod dva proizvoda iz kategorije Fridge Freezers, koje je model svrstao u Dishwashers.

Iako Linear SVC postiže visoku ukupnu tačnost od oko 97.2%, ovi primeri pokazuju da model može imati problem sa kratkim nazivima i oznakama modela koje ne sadrže dovoljno jasnih reči o vrsti proizvoda.

U sledećoj verziji bih pokušala da dodatno poboljšam model kroz podešavanje TF-IDF parametara i dodatni feature engineering.